# Imports

In [1]:
import optuna
import pickle

import numpy as np
import pandas as pd

from tqdm import tqdm

from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict

/home/junior/Documentos/GitHub/kaggle-competition-predicting-student-health/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Utils

In [2]:
def load_pickle(file_path):
    with open(file_path, 'rb') as file:
        return pickle.load(file)

# Loading Datasets

In [4]:
X_train = pd.read_parquet('../data/processed/X_train_stacking_layer_one_model_60_trials_data_type_feat_eng.parquet')
y_train = pd.read_parquet('../data/interim/y_train.parquet')

X_test = pd.read_parquet('../data/processed/X_test_stacking_layer_one_model_60_trials_data_type_feat_eng.parquet')

In [5]:
X_train.head()

,lgbm_0,lgbm_1,lgbm_2,xgb_0,xgb_1,xgb_2
0,0.003635,0.000557,0.995808,0.004425,0.000993,0.994582
1,0.995015,0.000736,0.004249,0.994982,0.000534,0.004484
2,0.003843,0.001026,0.995131,0.002812,0.000825,0.996363
3,0.003649,0.001574,0.994777,0.004159,0.001883,0.993958
4,0.995310,0.000026,0.004664,0.991897,0.000029,0.008074


In [6]:
X_test.head()

,lgbm_0,lgbm_1,lgbm_2,xgb_0,xgb_1,xgb_2
0,0.012695,0.003805,0.983500,0.010483,0.004324,0.985194
1,0.494675,0.003871,0.501454,0.506439,0.002756,0.490805
2,0.997004,0.002221,0.000775,0.997250,0.002230,0.000520
3,0.990545,0.000021,0.009434,0.989391,0.000017,0.010592
4,0.006499,0.001510,0.991991,0.009692,0.001897,0.988410


# Machine Learning

In [7]:
label_encoder = load_pickle('../artifacts/label_encoder.pkl')

In [8]:
def objective(trial, X, y):

    w0 = trial.suggest_float('weight_class_0', 0.001, 10.0, log=True)
    w1 = trial.suggest_float('weight_class_1', 0.001, 10.0, log=True)
    w2 = trial.suggest_float('weight_class_2', 0.001, 10.0, log=True)
    weights = np.array([w0, w1, w2])

    proba = X.loc[:, ['xgb_0', 'xgb_1', 'xgb_2']].to_numpy()
    
    weighted_probas = proba * weights
    pred = np.argmax(weighted_probas, axis=1)
    
    score = balanced_accuracy_score(y, pred)

    return score


study = optuna.create_study(
    direction="maximize", 
    sampler=optuna.samplers.TPESampler(seed=42), 
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=2)
)

study.optimize(
    lambda trial: objective(trial, X_train, y_train.health_condition), 
    n_trials=500, 
    n_jobs=-1, 
    show_progress_bar=True
)

print("\nBest trial score:")
print(study.best_trial.value)

print("\nBest params:")
print(study.best_trial.params)

[I 2026-07-13 11:33:48,231] A new study created in memory with name: no-name-ae4c531e-de4e-4930-afea-23b234ae6327
                                                                                                                                                                                                                   

[I 2026-07-13 11:33:48,393] Trial 9 finished with value: 0.5948266534833779 and parameters: {'weight_class_0': 0.7508109608873067, 'weight_class_1': 0.0017896934277486895, 'weight_class_2': 0.39374864154686906}. Best is trial 9 with value: 0.5948266534833779.
[I 2026-07-13 11:33:48,397] Trial 6 finished with value: 0.8666613453789284 and parameters: {'weight_class_0': 1.9951465472062095, 'weight_class_1': 0.1880216634308751, 'weight_class_2': 4.790734804918955}. Best is trial 6 with value: 0.8666613453789284.
[I 2026-07-13 11:33:48,398] Trial 10 finished with value: 0.8422306015053239 and parameters: {'weight_class_0': 0.05094593775052039, 'weight_class_1': 0.023035850791264738, 'weight_class_2': 0.013149627889062307}. Best is trial 6 with value: 0.8666613453789284.
[I 2026-07-13 11:33:48,406] Trial 4 finished with value: 0.7719207124132869 and parameters: {'weight_class_0': 0.5526022123125292, 'weight_class_1': 0.012180390259252424, 'weight_class_2': 0.06354383867694655}. Best is tria

[I 2026-07-13 11:33:48,558] Trial 13 finished with value: 0.76185593250411 and parameters: {'weight_class_0': 0.009764906410084156, 'weight_class_1': 0.8699526074133042, 'weight_class_2': 0.001043609058715541}. Best is trial 7 with value: 0.8917257026768143.
[I 2026-07-13 11:33:48,572] Trial 15 finished with value: 0.8323191437637307 and parameters: {'weight_class_0': 0.008614050893175513, 'weight_class_1': 0.5511961081061859, 'weight_class_2': 0.0012938115882767222}. Best is trial 7 with value: 0.8917257026768143.
[I 2026-07-13 11:33:48,576] Trial 14 finished with value: 0.9400701104744545 and parameters: {'weight_class_0': 0.008340046602435014, 'weight_class_1': 0.6930059377775365, 'weight_class_2': 0.1438796129863513}. Best is trial 14 with value: 0.9400701104744545.
[I 2026-07-13 11:33:48,594] Trial 16 finished with value: 0.8491212584797637 and parameters: {'weight_class_0': 0.007678778229679644, 'weight_class_1': 0.36079530132583254, 'weight_class_2': 0.0010010478919744447}. Best

Best trial: 34. Best value: 0.949567:   7%|█████████▋                                                                                                                             | 36/500 [00:00<00:08, 53.95it/s]

[I 2026-07-13 11:33:48,757] Trial 25 finished with value: 0.9417307940619241 and parameters: {'weight_class_0': 0.036854116631509434, 'weight_class_1': 2.855573749299493, 'weight_class_2': 0.37985814616467656}. Best is trial 22 with value: 0.949131320999292.
[I 2026-07-13 11:33:48,758] Trial 26 finished with value: 0.938409398343703 and parameters: {'weight_class_0': 0.03957119553150766, 'weight_class_1': 3.555346776911412, 'weight_class_2': 0.40640678166796956}. Best is trial 22 with value: 0.949131320999292.
[I 2026-07-13 11:33:48,778] Trial 27 finished with value: 0.9392307122394961 and parameters: {'weight_class_0': 0.04298272557342732, 'weight_class_1': 3.746271985022204, 'weight_class_2': 0.49910708650043956}. Best is trial 22 with value: 0.949131320999292.
[I 2026-07-13 11:33:48,792] Trial 28 finished with value: 0.936150255601269 and parameters: {'weight_class_0': 0.03081121636799222, 'weight_class_1': 2.9290046119283204, 'weight_class_2': 0.22513618996009305}. Best is trial 22

...

[I 2026-07-13 11:33:56,919] Trial 474 finished with value: 0.9491863620820137 and parameters: {'weight_class_0': 0.29083988907479197, 'weight_class_1': 2.8930745863157434, 'weight_class_2': 2.159248124365757}. Best is trial 457 with value: 0.9497733188665324.
[I 2026-07-13 11:33:56,964] Trial 476 finished with value: 0.9491640779776923 and parameters: {'weight_class_0': 0.2813876696065295, 'weight_class_1': 2.8262592747864574, 'weight_class_2': 2.0779544607438867}. Best is trial 457 with value: 0.9497733188665324.
[I 2026-07-13 11:33:56,965] Trial 475 finished with value: 0.947840945517575 and parameters: {'weight_class_0': 0.19601359599797438, 'weight_class_1': 3.0798596416034543, 'weight_class_2': 6.09792907045335}. Best is trial 457 with value: 0.9497733188665324.
[I 2026-07-13 11:33:56,978] Trial 477 finished with value: 0.9493602670406666 and parameters: {'weight_class_0': 0.19327797599273758, 'weight_class_1': 4.951963211081684, 'weight_class_2': 2.081397666532235}. Best is trial

Best trial: 457. Best value: 0.949773: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▍| 498/500 [00:09<00:00, 61.88it/s]

[I 2026-07-13 11:33:57,107] Trial 484 finished with value: 0.9484366023854274 and parameters: {'weight_class_0': 0.19106611765823867, 'weight_class_1': 4.327062897537358, 'weight_class_2': 1.1449168202780562}. Best is trial 457 with value: 0.9497733188665324.
[I 2026-07-13 11:33:57,135] Trial 485 finished with value: 0.9483775878748787 and parameters: {'weight_class_0': 0.1963393442090156, 'weight_class_1': 4.70150419663289, 'weight_class_2': 1.1864931351990498}. Best is trial 457 with value: 0.9497733188665324.
[I 2026-07-13 11:33:57,139] Trial 486 finished with value: 0.9493226637326871 and parameters: {'weight_class_0': 0.09819806686371707, 'weight_class_1': 2.501890288638061, 'weight_class_2': 1.1694954539313913}. Best is trial 457 with value: 0.9497733188665324.
[I 2026-07-13 11:33:57,189] Trial 487 finished with value: 0.9494116757964589 and parameters: {'weight_class_0': 0.10126806722765751, 'weight_class_1': 2.5093118263942897, 'weight_class_2': 0.9928647221728296}. Best is tri

Best trial: 457. Best value: 0.949773: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 500/500 [00:09<00:00, 54.76it/s]

[I 2026-07-13 11:33:57,361] Trial 499 finished with value: 0.9465854560221123 and parameters: {'weight_class_0': 0.0777290885987139, 'weight_class_1': 3.707566387098161, 'weight_class_2': 1.815596842335533}. Best is trial 457 with value: 0.9497733188665324.

Best trial score:
0.9497733188665324

Best params:
{'weight_class_0': 0.16158000859014726, 'weight_class_1': 3.024953878433461, 'weight_class_2': 1.6231917599188501}


In [9]:
weights = np.array([study.best_params['weight_class_0'], study.best_params['weight_class_1'], study.best_params['weight_class_2']])
weighted_probas = X_test.loc[:, ['xgb_0', 'xgb_1', 'xgb_2']].to_numpy() * weights

pred = np.argmax(weighted_probas, axis=1)

In [10]:
sub_labels = label_encoder.inverse_transform(pred)

# Submission

In [11]:
submission = pd.read_csv('../data/submission/sample_submission.csv')
submission['health_condition'] = sub_labels

submission.to_csv('../data/submission/submission_from_5.2_xgboost_submission.csv', index=False)

In [12]:
submission.head()

,id,health_condition
0,690088,unhealthy
1,690089,unhealthy
2,690090,at-risk
3,690091,at-risk
4,690092,unhealthy
